In [18]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import random
from collections import defaultdict

from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

# --- Config ---
DATASET_FOLDER = "Dataset"
NUM_EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3
VAL_PER_CLASS = 15  # images per class reserved for validation

# --- Device ---
# Use Apple MPS (Metal) if available, otherwise fall back to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# --- Dataset ---
# Load all images from the folder structure: Dataset/<class_name>/<image>
dataset = load_dataset("imagefolder", data_dir=DATASET_FOLDER)
num_classes = len(dataset['train'].features['label'].names)
print(f"Number of classes: {num_classes}")

# --- Transforms ---
# Training: random crop + flip for data augmentation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),   # randomly crop and resize to 224x224
    transforms.RandomHorizontalFlip(),   # random mirror for augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet stats
])

# Validation: deterministic center crop, no augmentation
val_transforms = transforms.Compose([
    transforms.Resize(256),              # resize shorter side to 256
    transforms.CenterCrop(224),          # crop center 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet stats
])

# --- Dataset wrapper ---
# Wraps a HuggingFace dataset split with a list of indices and applies transforms
class PlantDataset(Dataset):
    def __init__(self, hf_dataset, indices, transform=None):
        self.dataset = hf_dataset
        self.indices = indices  # subset of indices into hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        item = self.dataset[self.indices[idx]]
        image = item['image'].convert('RGB')
        label = item['label']
        if self.transform:
            image = self.transform(image)
        return image, label

# --- Stratified train/val split ---
# For each class, reserve VAL_PER_CLASS images for validation and use the rest for training.
# This ensures every class is represented in the val set.
all_labels = dataset['train']['label']
class_indices = defaultdict(list)
for idx, label in enumerate(all_labels):
    class_indices[label].append(idx)

train_indices, val_indices = [], []
for label, indices in class_indices.items():
    random.shuffle(indices)
    val_indices.extend(indices[:VAL_PER_CLASS])    # first VAL_PER_CLASS go to val
    train_indices.extend(indices[VAL_PER_CLASS:])  # rest go to train

train_dataset = PlantDataset(dataset['train'], train_indices, transform=train_transforms)
val_dataset = PlantDataset(dataset['train'], val_indices, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)} ({VAL_PER_CLASS} per class)")

# --- Model ---
# Load ResNet50 pretrained on ImageNet, then replace the final layer for our classes
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Freeze all layers — we only want to train the new classification head
for param in model.parameters():
    param.requires_grad = False

# Replace FC: ResNet50's default output is 1000 classes → swap for num_classes
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)
print(f"Model ready. FC: {model.fc}")

# --- Training ---
criterion = nn.CrossEntropyLoss()
# Only pass FC parameters to the optimizer since all other layers are frozen
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)

for epoch in range(NUM_EPOCHS):
    # --- Train phase ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()          # clear gradients from previous step
        outputs = model(images)        # forward pass
        loss = criterion(outputs, labels)
        loss.backward()                # compute gradients
        optimizer.step()               # update FC weights

        running_loss += loss.item()
        _, predicted = outputs.max(1)  # class with highest logit
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100.0 * correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

    # --- Validation phase ---
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():  # no gradients needed during eval
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    val_acc = 100.0 * val_correct / val_total
    print(f"           Val Acc: {val_acc:.2f}%")

# --- Save model ---
torch.save(model.state_dict(), "resnet50_plants.pth")
print("Model saved to resnet50_plants.pth")


Using device: mps
Number of classes: 47
Train samples: 14085
Val samples:   705 (15 per class)
Model ready. FC: Linear(in_features=2048, out_features=47, bias=True)


Epoch 1/10: 100%|██████████| 441/441 [04:37<00:00,  1.59it/s]


Epoch [1/10] Loss: 1.7650 | Train Acc: 63.36%
           Val Acc: 76.31%


Epoch 2/10: 100%|██████████| 441/441 [04:57<00:00,  1.48it/s]


Epoch [2/10] Loss: 0.8838 | Train Acc: 80.03%
           Val Acc: 82.70%


...

Epoch [8/10] Loss: 0.4328 | Train Acc: 88.80%
           Val Acc: 87.80%


Epoch 9/10: 100%|██████████| 441/441 [05:44<00:00,  1.28it/s]


Epoch [9/10] Loss: 0.4075 | Train Acc: 89.50%
           Val Acc: 87.94%


Epoch 10/10: 100%|██████████| 441/441 [06:04<00:00,  1.21it/s]


Epoch [10/10] Loss: 0.3905 | Train Acc: 90.04%
           Val Acc: 86.38%
Model saved to resnet50_plants.pth
